# Diff-Image Displacement Predictor — Robust Synthetic 10x Double-Overlay Data

Same model, training, and eval as `diff_pred_20x.ipynb` (blank ViT + MLP head on
`diff = frame_j - frame_i`). The data pipeline is different: instead of SIFT-aligning a
fixed bottom/top design onto a real video, every training pair is built from **two randomly picked
10x-magnification still images**. The "bottom" is a single static reference (resized once, then
center-cropped to 224x224). The "top" is resized once to a 448x448 square, then two random 224x224
windows are cropped from it (at most 100px apart in x/y) and each is blended onto the shared bottom
crop with independently randomized opacity/blur/contrast — the model predicts the displacement
between those two windows. This pre-resize + direct-window-crop approach mimics the ~2x zoom that
would make 10x content match a 20x field of view, so this data (and a model trained on it) should
transfer to real 20x recordings.

### Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import multiprocessing
import warnings
import numpy as np
import cv2
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
from transformers import ViTModel, ViTConfig

multiprocessing.set_start_method('fork', force=True)
warnings.filterwarnings('ignore')

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'model_notebooks' else Path.cwd()
CKPT_DIR = REPO_ROOT / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'device: {device}')

## Data generation
### Image pool: collect all 10x stills from `FOLDERS`, split 80/20 into train/val

In [ ]:
# TODO: fill in with the folders of 10x-magnification stills — every image file directly inside
# each folder is added to the pool (not recursive).
FOLDERS = [
    # REPO_ROOT / '10X_stills' / '0',
    # REPO_ROOT / '10X_stills' / '1',
]

IMG_SIZE   = 224     # matches diff_pred_20x.ipynb's ViT input size
MAX_CLIP   = 60      # diff visualization clip only (sanity-check cell) — matches diff_pred_20x.ipynb

TOP_OPACITY_RANGE = (0.15, 0.25)  # one value sampled per pair, shared by both overlay composites

# augmentation blur applied while compositing — top and bottom draw from separate ranges (top is
# offset 2 higher than bottom, e.g. modeling it sitting further out of focus), 2 independent samples
# each per pair (composite 1/2). This is the only place blurring happens; the model itself
# (extract_embedding) never blurs anything.
BLUR_SIGMA_BOTTOM_RANGE = (1.0, 3.0)
BLUR_SIGMA_TOP_RANGE    = (3.0, 5.0)

CONTRAST_RANGE = (0.7, 1.3)  # augmentation contrast applied to top/bottom while compositing —
                              # same sampling pattern as the blur ranges (4 independent samples per
                              # pair); 1.0 = unchanged contrast

BRIGHTNESS_RANGE = (-30, 30)  # augmentation brightness (additive) applied to top/bottom while
                               # compositing — same sampling pattern as CONTRAST_RANGE (4
                               # independent samples per pair); 0 = unchanged brightness

# pre-resize + window-crop sizes (reduces per-sample compute vs. compositing at full resolution).
# bottom is a single static reference: resized once, then center-cropped to IMG_SIZE.
BOTTOM_RESIZE_W = 448
BOTTOM_RESIZE_H = 488
# top is resized once to a square, then two IMG_SIZE windows are cropped from it at random
# positions no more than TOP_WINDOW_MAX_OFFSET px (in the resized 448 space) apart in x or y.
TOP_RESIZE = 448
TOP_WINDOW_MAX_OFFSET = 100

SAMPLES_PER_EPOCH     = 5000
SAMPLES_PER_EPOCH_VAL = 1000
TRAIN_FRACTION = 0.8
SPLIT_SEED     = 0

AXIS_NAMES = ['x', 'y']
AXIS_UNITS = ['mm', 'mm']

# px-per-mm: video_overlay_align.py only measured this at 5x and scales it proportionally with
# magnification for other zoom levels (RATIO_20X = RATIO_5X * 4, same convention used in
# diff_pred_20x.ipynb). 10x is 2x the magnification of 5x (half that of 20x), so:
#   RATIO_10X_PX_PER_MM = RATIO_5X_PX_PER_MM * 2.0 == RATIO_20X_PX_PER_MM / 2.0
# NOTE: this is the physically-consistent reading of "10x has double the px/mm ratio of 20x" — i.e.
# 10x is *lower* magnification than 20x, so its px/mm ratio is smaller (half), not double. Flag if
# a literal "10x ratio = 2x the 20x ratio" was intended instead.
# NOTE: calibrated in *native* (full-resolution) px, not the resized 448 window space above — target
# computation in make_pair_sample rescales window-space px back to native px before dividing by this.
RATIO_5X_PX_PER_MM  = np.array([-72.0, 150.0], dtype=np.float32)
RATIO_20X_PX_PER_MM = RATIO_5X_PX_PER_MM * 4.0
RATIO_10X_PX_PER_MM = RATIO_20X_PX_PER_MM / 2.0

IMAGE_EXTS = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff')


def collect_images(folders):
    paths = []
    for folder in folders:
        folder = Path(folder)
        for ext in IMAGE_EXTS:
            paths.extend(sorted(folder.glob(ext)))
    return paths


all_images = collect_images(FOLDERS)
print(f'Found {len(all_images):,} images across {len(FOLDERS)} folders')

_rng_split = np.random.default_rng(SPLIT_SEED)
_perm = _rng_split.permutation(len(all_images))
_n_train = int(round(TRAIN_FRACTION * len(all_images)))
train_paths = [all_images[i] for i in _perm[:_n_train]]
val_paths   = [all_images[i] for i in _perm[_n_train:]]
print(f'Train: {len(train_paths):,} images   Val: {len(val_paths):,} images')


### Image cache

Reading full-resolution (e.g. 3840x2160) stills from Google Drive on every `__getitem__` call is
the dominant cost when Drive is mounted in Colab — each read is a slow network round trip, repeated
every sample, every epoch, for a pool of images that never changes. Instead, every pool image is
read from disk exactly once here and immediately reduced to the two small arrays `make_pair_sample`
actually needs (the static `IMG_SIZE` bottom crop, and the `TOP_RESIZE` square for top-window
cropping) — cached in RAM, keyed by path. From then on, sampling only slices/blurs/blends
already-in-memory small arrays; no further disk or Drive I/O happens during training.

In [ ]:
def _cache_image(path: Path):
    """Read `path` from disk exactly once, return the two derived arrays make_pair_sample needs:
    the static IMG_SIZE bottom crop, and the TOP_RESIZE square used for top-window cropping
    (plus the image's native (w, h), needed to rescale window px back to native px for target_mm).
    """
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(f'Could not read: {path}')
    h, w = img.shape[:2]

    bottom_resized = cv2.resize(img, (BOTTOM_RESIZE_W, BOTTOM_RESIZE_H), interpolation=cv2.INTER_AREA)
    bh, bw = bottom_resized.shape[:2]
    by0, bx0 = (bh - IMG_SIZE) // 2, (bw - IMG_SIZE) // 2
    bottom_crop = bottom_resized[by0:by0 + IMG_SIZE, bx0:bx0 + IMG_SIZE].copy()

    top_resized = cv2.resize(img, (TOP_RESIZE, TOP_RESIZE), interpolation=cv2.INTER_AREA)

    return bottom_crop, top_resized, (w, h)


BOTTOM_CROP_CACHE, TOP_RESIZED_CACHE, TOP_NATIVE_SIZE_CACHE = {}, {}, {}
for _p in tqdm(all_images, desc='Caching images'):
    _bottom_crop, _top_resized, _native_size = _cache_image(_p)
    BOTTOM_CROP_CACHE[_p] = _bottom_crop
    TOP_RESIZED_CACHE[_p] = _top_resized
    TOP_NATIVE_SIZE_CACHE[_p] = _native_size

_cache_mb = (sum(a.nbytes for a in BOTTOM_CROP_CACHE.values()) +
             sum(a.nbytes for a in TOP_RESIZED_CACHE.values())) / 1e6
print(f'Cached {len(all_images):,} images in RAM ({_cache_mb:.1f} MB total)')


### Compositing helpers

`make_pair_sample` picks 2 distinct images from a pool ("top", "bottom"). The bottom is resized
once to `(BOTTOM_RESIZE_W, BOTTOM_RESIZE_H)` and center-cropped to `IMG_SIZE` — a single static
crop shared by both composites. The top is resized once to a `TOP_RESIZE` square, then two
`IMG_SIZE` windows are cropped from it at random positions at most `TOP_WINDOW_MAX_OFFSET` px
apart in x and y (rejection-sampled). Each window is blended onto the shared bottom crop with its
own randomly sampled blur sigma (top drawn from `BLUR_SIGMA_TOP_RANGE`, bottom from
`BLUR_SIGMA_BOTTOM_RANGE`, offset 2 higher for top), its own randomly sampled top/bottom contrast
factor, and its own randomly sampled top/bottom brightness delta, both composites sharing one
randomly sampled opacity. The target is the displacement between the two window positions,
rescaled from the resized `TOP_RESIZE` space back to native px and converted from px to mm via
`RATIO_10X_PX_PER_MM`.


In [ ]:
def blur_img(img: np.ndarray, sigma: float) -> np.ndarray:
    if sigma <= 0:
        return img
    k = 2 * max(1, int(round(3 * sigma))) + 1
    return cv2.GaussianBlur(img, (k, k), sigma)


def adjust_contrast(img: np.ndarray, factor: float) -> np.ndarray:
    """Scale contrast around mid-gray (127.5); factor=1.0 is unchanged, <1 flattens, >1 boosts."""
    return cv2.convertScaleAbs(img, alpha=factor, beta=127.5 * (1.0 - factor))


def adjust_brightness(img: np.ndarray, delta: float) -> np.ndarray:
    """Additive brightness shift; delta=0 is unchanged, <0 darkens, >0 brightens."""
    return cv2.convertScaleAbs(img, alpha=1.0, beta=delta)


def make_composite(top_window: np.ndarray, bottom_crop: np.ndarray,
                    opacity: float, sigma_top: float, sigma_bottom: float,
                    contrast_top: float, contrast_bottom: float,
                    brightness_top: float, brightness_bottom: float) -> np.ndarray:
    """Blur + contrast + brightness-adjust `top_window`/`bottom_crop` (already same-sized IMG_SIZE
    crops) independently, then blend."""
    top_b    = adjust_brightness(adjust_contrast(blur_img(top_window, sigma_top), contrast_top), brightness_top)
    bottom_b = adjust_brightness(adjust_contrast(blur_img(bottom_crop, sigma_bottom), contrast_bottom), brightness_bottom)
    return cv2.addWeighted(top_b, opacity, bottom_b, 1.0 - opacity, 0.0)


def make_pair_sample(bottom_path: Path, top_path: Path, rng: np.random.Generator):
    """Returns (frame_i, frame_j, target_mm, debug):
    bottom_path/top_path are looked up in BOTTOM_CROP_CACHE/TOP_RESIZED_CACHE (precomputed once,
    see the image cache cell above) — no disk/Drive reads happen here. bottom's cached crop is a
    single static reference shared by both composites. top's cached TOP_RESIZE square has two
    IMG_SIZE windows cropped from random positions at most TOP_WINDOW_MAX_OFFSET px apart in x and
    y (rejection-sampled). Each composite blurs/contrasts/brightens its own top window + the shared
    bottom crop independently, then blends them (same random opacity for both).
    frame_i/frame_j = the two composites.
    target_mm = [Δx, Δy] mm — how far the top content moved from frame_i's window to frame_j's,
    rescaled from the resized 448 window space to native px before converting via
    RATIO_10X_PX_PER_MM (moving the crop window +x is equivalent to the content shifting -x).
    debug = dict of the sampled opacity/sigmas/contrasts/brightnesses/window positions, for the
    sanity-check cell.
    """
    bottom_crop = BOTTOM_CROP_CACHE[bottom_path]
    top_resized = TOP_RESIZED_CACHE[top_path]
    top_w_orig, top_h_orig = TOP_NATIVE_SIZE_CACHE[top_path]

    max_pos = TOP_RESIZE - IMG_SIZE
    x1, y1 = rng.integers(0, max_pos + 1), rng.integers(0, max_pos + 1)
    while True:
        x2, y2 = rng.integers(0, max_pos + 1), rng.integers(0, max_pos + 1)
        if abs(x2 - x1) <= TOP_WINDOW_MAX_OFFSET and abs(y2 - y1) <= TOP_WINDOW_MAX_OFFSET:
            break

    top_window_1 = top_resized[y1:y1 + IMG_SIZE, x1:x1 + IMG_SIZE]
    top_window_2 = top_resized[y2:y2 + IMG_SIZE, x2:x2 + IMG_SIZE]

    opacity = float(rng.uniform(*TOP_OPACITY_RANGE))
    sigma_top_1, sigma_top_2 = rng.uniform(BLUR_SIGMA_TOP_RANGE[0], BLUR_SIGMA_TOP_RANGE[1], size=2)
    sigma_bottom_1, sigma_bottom_2 = rng.uniform(BLUR_SIGMA_BOTTOM_RANGE[0], BLUR_SIGMA_BOTTOM_RANGE[1], size=2)
    contrast_top_1, contrast_bottom_1, contrast_top_2, contrast_bottom_2 = rng.uniform(
        CONTRAST_RANGE[0], CONTRAST_RANGE[1], size=4)
    brightness_top_1, brightness_bottom_1, brightness_top_2, brightness_bottom_2 = rng.uniform(
        BRIGHTNESS_RANGE[0], BRIGHTNESS_RANGE[1], size=4)

    frame_i = make_composite(top_window_1, bottom_crop, opacity, sigma_top_1, sigma_bottom_1,
                              contrast_top_1, contrast_bottom_1, brightness_top_1, brightness_bottom_1)
    frame_j = make_composite(top_window_2, bottom_crop, opacity, sigma_top_2, sigma_bottom_2,
                              contrast_top_2, contrast_bottom_2, brightness_top_2, brightness_bottom_2)

    # window-space (448-resized) px -> native px, before applying RATIO_10X_PX_PER_MM (calibrated
    # in native px). Moving the crop window by +Δ shows content shifted by -Δ, so negate.
    scale_x, scale_y = top_w_orig / TOP_RESIZE, top_h_orig / TOP_RESIZE
    target_px = np.array([(x1 - x2) * scale_x, (y1 - y2) * scale_y], dtype=np.float32)
    target_mm = (target_px / RATIO_10X_PX_PER_MM).astype(np.float32)

    debug = dict(opacity=opacity, sigma_top_1=sigma_top_1, sigma_bottom_1=sigma_bottom_1,
                 sigma_top_2=sigma_top_2, sigma_bottom_2=sigma_bottom_2,
                 contrast_top_1=contrast_top_1, contrast_bottom_1=contrast_bottom_1,
                 contrast_top_2=contrast_top_2, contrast_bottom_2=contrast_bottom_2,
                 brightness_top_1=brightness_top_1, brightness_bottom_1=brightness_bottom_1,
                 brightness_top_2=brightness_top_2, brightness_bottom_2=brightness_bottom_2,
                 window_1=(x1, y1), window_2=(x2, y2))
    return frame_i, frame_j, target_mm, debug


### Dataset

Each item picks 2 distinct images at random from the split's pool and calls `make_pair_sample`.

In [ ]:
class SyntheticOverlayDisplacementDataset(Dataset):
    """Each item: 2 distinct random images from `image_paths` ("top", "bottom"), composited via
    make_pair_sample (bottom center-cropped once, top window-cropped twice at nearby random
    positions). Target is the [Δx, Δy] mm displacement between the two top windows."""

    def __init__(self, image_paths, n_samples, seed=None):
        self.image_paths = image_paths
        self.n_samples = n_samples
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        i1, i2 = self.rng.choice(len(self.image_paths), size=2, replace=False)
        bottom_path, top_path = self.image_paths[i1], self.image_paths[i2]
        frame_i, frame_j, target_mm, _debug = make_pair_sample(bottom_path, top_path, self.rng)

        frame_i_t = torch.from_numpy(frame_i[:, :, ::-1].copy()).permute(2, 0, 1)   # BGR -> RGB, (3, H, W) uint8
        frame_j_t = torch.from_numpy(frame_j[:, :, ::-1].copy()).permute(2, 0, 1)
        target_t  = torch.from_numpy(target_mm)
        return frame_i_t, frame_j_t, target_t


def _worker_init_fn(_worker_id):
    # each forked worker inherits the parent's rng state — reseed per worker so they don't all
    # generate the same stream of synthetic samples.
    info = torch.utils.data.get_worker_info()
    info.dataset.rng = np.random.default_rng(info.seed % (2 ** 32))


train_dataset = SyntheticOverlayDisplacementDataset(train_paths, SAMPLES_PER_EPOCH)
train_loader  = DataLoader(train_dataset, batch_size=32, num_workers=4, drop_last=True, worker_init_fn=_worker_init_fn)
print(f'Train dataset size: {len(train_dataset):,} pairs/epoch')

val_dataset = SyntheticOverlayDisplacementDataset(val_paths, SAMPLES_PER_EPOCH_VAL, seed=1234)
val_loader  = DataLoader(val_dataset, batch_size=32, num_workers=0, drop_last=False)
print(f'Val dataset size  : {len(val_dataset):,} pairs')

### Sanity check — dataloader

Pulls one item from `train_dataset`, visualizes `frame_i`/`frame_j`/the diff magnitude heatmap, and
prints the sampled window positions/opacity/blur sigmas/contrasts alongside the resulting target —
run this before training to confirm the compositing/cropping/target computation all look right.

In [ ]:
# Call make_pair_sample directly (the same call train_dataset.__getitem__ makes) so we can also
# print the sampled offsets/opacity/blur sigmas, which __getitem__ itself doesn't return.
_rng_sanity = np.random.default_rng()
_bi, _ti = _rng_sanity.choice(len(train_dataset.image_paths), size=2, replace=False)
_bottom_path, _top_path = train_dataset.image_paths[_bi], train_dataset.image_paths[_ti]
sanity_frame_i, sanity_frame_j, sanity_target, sanity_debug = make_pair_sample(_bottom_path, _top_path, _rng_sanity)

diff_vis = np.abs(sanity_frame_j.astype(np.float32) - sanity_frame_i.astype(np.float32)).mean(axis=-1)
diff_vis_clipped = np.clip(diff_vis, 0, MAX_CLIP)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cv2.cvtColor(sanity_frame_i, cv2.COLOR_BGR2RGB)); axes[0].set_title('frame_i (composite 1)'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(sanity_frame_j, cv2.COLOR_BGR2RGB)); axes[1].set_title('frame_j (composite 2)'); axes[1].axis('off')
im = axes[2].imshow(diff_vis_clipped, cmap='inferno', vmin=0, vmax=MAX_CLIP); axes[2].set_title(f'|diff| clipped at {MAX_CLIP}'); axes[2].axis('off')
fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

target_str = '  '.join(f'{n}={sanity_target[i]:+.4f}{u}' for i, (n, u) in enumerate(zip(AXIS_NAMES, AXIS_UNITS)))
debug_str = (f"opacity={sanity_debug['opacity']:.3f}  "
             f"sigma_top=({sanity_debug['sigma_top_1']:.2f}, {sanity_debug['sigma_top_2']:.2f})  "
             f"sigma_bottom=({sanity_debug['sigma_bottom_1']:.2f}, {sanity_debug['sigma_bottom_2']:.2f})\n"
             f"contrast_top=({sanity_debug['contrast_top_1']:.2f}, {sanity_debug['contrast_top_2']:.2f})  "
             f"contrast_bottom=({sanity_debug['contrast_bottom_1']:.2f}, {sanity_debug['contrast_bottom_2']:.2f})\n"
             f"brightness_top=({sanity_debug['brightness_top_1']:.2f}, {sanity_debug['brightness_top_2']:.2f})  "
             f"brightness_bottom=({sanity_debug['brightness_bottom_1']:.2f}, {sanity_debug['brightness_bottom_2']:.2f})\n"
             f"window_1={sanity_debug['window_1'][0]},{sanity_debug['window_1'][1]} px   "
             f"window_2={sanity_debug['window_2'][0]},{sanity_debug['window_2'][1]} px")
plt.suptitle(f'bottom={_bottom_path.name}  top={_top_path.name}\n{debug_str}\ntarget: {target_str}', fontsize=9)
plt.tight_layout()
plt.show()


### Sanity check — synthetic 10x composite vs. a real 20x video frame

Cross-check that the synthetic compositing pipeline (built from 10x stills) actually lines up with a
real 20x capture once properly registered. Uses only `20X_DropDown/5` (not the general `FOLDERS`
pool). `bottom_10x.jpg` is SIFT-aligned directly (no manual pre-scaling) onto a random frame of
`20X_DropDown/5/20x.mp4` (same `sift_affine` pipeline as `diff_pred_20x.ipynb` / `flake_designer.py`)
— there's no still actually named `20x.jpg` in this repo, so that video frame stands in for it. SIFT
keypoints are scale-invariant, so the fitted affine itself absorbs the ~2x scale gap between 10x and
20x magnification (no separate upscale step needed); `top_10x` is placed via that same fixed affine,
offset by a random (dx, dy) sampled in `bottom_10x`'s own pixel space (±1/4 of its height/width) —
this cell composites manually (translate + blend) rather than via `make_pair_sample`'s window-crop
approach, so its sampling scheme is independent of that cell's. Each of top/bottom is independently
blurred (top from `BLUR_SIGMA_TOP_RANGE`, bottom from `BLUR_SIGMA_BOTTOM_RANGE`, same ranges as the
dataloader) and blended at the same `TOP_OPACITY_RANGE`, all after SIFT/warping is done. Resizing —
both the composite and the real video frame down to `IMG_SIZE` — also only happens at the very end,
after SIFT.


In [ ]:
FOLDER_5 = Path('/content/drive/MyDrive/5')

# ---- load the two 10x stills + a random reference frame from the 20x video ----
_bottom_10x = cv2.imread(str(FOLDER_5 / 'bottom_10x.jpg'))
_top_10x_full = cv2.imread(str(FOLDER_5 / 'top_10x.jpg'))
_h10, _w10 = _bottom_10x.shape[:2]
_top_10x = cv2.resize(_top_10x_full, (_w10, _h10), interpolation=cv2.INTER_AREA)

_rng_20x_check = np.random.default_rng()

_cap_5 = cv2.VideoCapture(str(FOLDER_5 / '20x.mp4'))
_vw5, _vh5 = int(_cap_5.get(cv2.CAP_PROP_FRAME_WIDTH)), int(_cap_5.get(cv2.CAP_PROP_FRAME_HEIGHT))
_n_frames_5 = int(_cap_5.get(cv2.CAP_PROP_FRAME_COUNT))
_frame_idx_5 = int(_rng_20x_check.integers(0, _n_frames_5))
_cap_5.set(cv2.CAP_PROP_POS_FRAMES, _frame_idx_5)
_ok_5, _ref_frame_5 = _cap_5.read()
_cap_5.release()
if not _ok_5:
    raise RuntimeError(f'Could not read frame {_frame_idx_5} from {FOLDER_5 / "20x.mp4"}')

# ---- warp bottom_10x into the video's native frame via a precomputed SIFT affine ----
# (fit once with sift_affine(bottom_10x, ref_frame) and hardcoded here — the fitted affine
# absorbs the ~2x scale gap between 10x and 20x magnification, so no manual rescale is needed)
_M5 = np.array([
    [ 2.01572162e+00,  2.16065581e-02, -1.81823928e+03],
    [ 1.12195976e-03,  2.04328578e+00, -1.39008233e+03],
])
_A5, _t5 = _M5[:, :2], _M5[:, 2]
warped_bottom_native_5 = cv2.warpAffine(_bottom_10x, _M5, (_vw5, _vh5))

# ---- place top_10x via the same affine, offset by a random (dx, dy) ----
# offset sampled in bottom_10x's own pixel space, same bounds as make_pair_sample (±1/4 height,
# ±1/4 width), mapped through the affine's linear part into the video's native frame
# (same pattern as diff_pred_20x.ipynb: native_delta_px = A @ offset)
_dx10, _dy10 = _rng_20x_check.uniform(-_w10 / 4, _w10 / 4), _rng_20x_check.uniform(-_h10 / 4, _h10 / 4)
_offset_10 = np.array([_dx10, _dy10], dtype=np.float32)
_t_top_native_5 = _t5 + _A5 @ _offset_10
_M_top_native_5 = np.hstack([_A5, _t_top_native_5.reshape(2, 1)]).astype(np.float32)
warped_top_native_5 = cv2.warpAffine(_top_10x, _M_top_native_5, (_vw5, _vh5))

# ---- blur + blend, same augmentation ranges as the dataloader (top and bottom draw from
# separate blur ranges — top is offset 2 higher than bottom) ----
_opacity_5 = float(_rng_20x_check.uniform(*TOP_OPACITY_RANGE))
_sigma_top_5 = float(_rng_20x_check.uniform(*BLUR_SIGMA_TOP_RANGE))
_sigma_bottom_5 = float(_rng_20x_check.uniform(*BLUR_SIGMA_BOTTOM_RANGE))
_composite_native_5 = cv2.addWeighted(blur_img(warped_top_native_5, _sigma_top_5), _opacity_5,
                                       blur_img(warped_bottom_native_5, _sigma_bottom_5), 1.0 - _opacity_5, 0.0)

# ---- resize to IMG_SIZE ----
_synthetic_5 = cv2.resize(_composite_native_5, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
_real_20x_resized_5 = cv2.resize(_ref_frame_5, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

# ---- diff magnitude (RGB) ----
_diff_vis_5 = np.abs(_synthetic_5.astype(np.float32) - _real_20x_resized_5.astype(np.float32)).mean(axis=-1)
_diff_vis_5_clipped = np.clip(_diff_vis_5, 0, MAX_CLIP)

# ---- visualization ----
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cv2.cvtColor(_synthetic_5, cv2.COLOR_BGR2RGB)); axes[0].set_title('synthetic composite (SIFT-aligned onto video)'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(_real_20x_resized_5, cv2.COLOR_BGR2RGB)); axes[1].set_title(f'real 20x.mp4 frame {_frame_idx_5}'); axes[1].axis('off')
im5 = axes[2].imshow(_diff_vis_5_clipped, cmap='inferno', vmin=0, vmax=MAX_CLIP); axes[2].set_title(f'|diff| clipped at {MAX_CLIP}'); axes[2].axis('off')
fig.colorbar(im5, ax=axes[2], fraction=0.046, pad=0.04)

_debug_str_5 = (f'opacity={_opacity_5:.3f}  sigma_top={_sigma_top_5:.2f}  sigma_bottom={_sigma_bottom_5:.2f}\n'
                f'offset (bottom_10x space)=({_dx10:+.1f}, {_dy10:+.1f}) px   frame={_frame_idx_5}')
plt.suptitle(f'folder 5: SIFT-aligned synthetic composite vs. real 20x video frame\n{_debug_str_5}', fontsize=9)
plt.tight_layout()
plt.show()


## Model
diff -> blank ViT -> MLP head architecture (blur augmentation happens only in the dataloader, not
here — the model itself never blurs), with `TARGET_DIM=2` (`[Δx, Δy]` only — no ring-based
`Δcx, Δcy, Δarea` here).


In [ ]:
EMB_DIM    = 192    # blank ViT hidden size == CLS token dim
TARGET_DIM = 2        # [Δx, Δy]

vit_cfg = ViTConfig(
    num_channels=3, image_size=IMG_SIZE, patch_size=16,
    hidden_size=EMB_DIM, num_hidden_layers=6,
    num_attention_heads=3, intermediate_size=768,
)
vit = ViTModel(vit_cfg, add_pooling_layer=False).to(device)
vit.train()   # randomly initialized ('blank'), fully trainable

n_vit = sum(p.numel() for p in vit.parameters())
print(f'ViT params (all trainable, randomly initialized): {n_vit:,}')


def to_float(x):
    """(B, 3, H, W) uint8 -> (B, 3, H, W) float32 [0,1] on device."""
    return x.to(device, dtype=torch.float32).div_(255.0)


def extract_embedding(frame_i_float, frame_j_float):
    """(B, 3, 224, 224) float32 [0,1] x2 -> (B, EMB_DIM) CLS token of the blank ViT run on
    the difference image frame_j - frame_i. No blurring here — all blur augmentation happens
    in the dataloader (make_composite), never in the model itself.
    """
    diff = frame_j_float - frame_i_float                     # (B, 3, H, W)
    hidden = vit(diff, interpolate_pos_encoding=False).last_hidden_state
    return hidden[:, 0]                                      # (B, EMB_DIM) CLS token


class DisplacementHead(nn.Module):
    """CLS token -> small MLP -> (2,): [Δx, Δy]. Kept under a 100k-param budget."""

    def __init__(self, emb_dim=EMB_DIM, target_dim=TARGET_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, 256),
            nn.SiLU(),
            nn.Linear(256, target_dim),
        )

    def forward(self, emb):
        return self.net(emb)


def sigreg_loss(emb, gamma=1.0, eps=1e-4):
    """Variance/'sigreg' hinge loss: penalize embedding dimensions whose batch-wise std
    falls below gamma, to prevent representation collapse while training the encoder.
    """
    std = torch.sqrt(emb.var(dim=0) + eps)
    return F.relu(gamma - std).mean()


head = DisplacementHead(emb_dim=EMB_DIM, target_dim=TARGET_DIM).to(device)
n_head = sum(p.numel() for p in head.parameters())
print(f'Head params (trainable): {n_head:,}')
assert n_head <= 100_000, f'head has {n_head:,} params, exceeds the 100k budget'

with torch.no_grad():
    _fi   = torch.zeros(2, 3, IMG_SIZE, IMG_SIZE, device=device)
    _fj   = torch.zeros(2, 3, IMG_SIZE, IMG_SIZE, device=device)
    _emb  = extract_embedding(_fi, _fj)
    _out  = head(_emb)
    print(f'Embedding shape : {_emb.shape}')   # expect (2, 192)
    print(f'Output shape    : {_out.shape}')   # expect (2, 2)


### Target normalization
Recomputed from actual `train_dataset` samples (mirrors `diff_pred_20x.ipynb`'s approach), simplified
since `[Δx, Δy]` is always valid here — no ring-based masking needed.

In [ ]:
_stats_dataset = SyntheticOverlayDisplacementDataset(train_paths, n_samples=6000, seed=42)

_samples = []
for i in range(len(_stats_dataset)):
    _, _, target = _stats_dataset[i]
    _samples.append(target.numpy())
_samples = np.stack(_samples)   # (N, 2)

delta_mean = torch.from_numpy(_samples.mean(0).astype(np.float32)).to(device)
delta_std  = torch.from_numpy(_samples.std(0).clip(min=1e-3).astype(np.float32)).to(device)

print(f'Target stats from {len(_stats_dataset):,} train_dataset samples:')
for i, (name, unit) in enumerate(zip(AXIS_NAMES, AXIS_UNITS)):
    print(f'  {name}: mean={delta_mean[i]:.4f}  std={delta_std[i]:.4f} {unit}')


def normalize_target(target):
    """(B, 2) raw [Δx, Δy] -> (B, 2) z-scored."""
    return (target - delta_mean) / delta_std


def denormalize_target(pred):
    """(B, 2) z-scored -> (B, 2) raw [Δx, Δy]."""
    return pred * delta_std + delta_mean

## Training
The whole ViT is trainable, optimized jointly with the head — trained from scratch (no warm-start
checkpoint, unlike `diff_pred_20x.ipynb`, since this head's output dim (2) is incompatible with any
existing 5-dim checkpoint).

In [ ]:
STARTING_EPOCH = 0
NUM_EPOCHS     = STARTING_EPOCH + 50
SIGREG_WEIGHT  = 0.05
SIGREG_GAMMA   = 1.0

optimizer = torch.optim.AdamW([
    {'params': vit.parameters(),  'lr': 1e-4},
    {'params': head.parameters(), 'lr': 1e-3},
], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - STARTING_EPOCH)


def huber_loss_fn(pred, target_norm, delta=1.0):
    return F.huber_loss(pred, target_norm, delta=delta, reduction='mean')


def abs_error_sum(pred_raw, target):
    """Returns (sum of |error| per axis, count of entries) for accumulation across batches."""
    abs_err = (pred_raw - target).abs()
    return abs_err.sum(dim=0).cpu(), target.shape[0]


def run_validation():
    vit.eval(); head.eval()
    losses = []
    err_sum, err_count = torch.zeros(TARGET_DIM), 0
    with torch.no_grad():
        for frame_i, frame_j, target in val_loader:
            frame_i, frame_j = to_float(frame_i), to_float(frame_j)
            target = target.to(device)
            target_norm = normalize_target(target)

            emb  = extract_embedding(frame_i, frame_j)
            pred = head(emb)
            loss = huber_loss_fn(pred, target_norm)

            pred_raw = denormalize_target(pred)
            e_sum, e_cnt = abs_error_sum(pred_raw, target)
            err_sum += e_sum; err_count += e_cnt
            losses.append(loss.item())
    vit.train(); head.train()
    return float(np.mean(losses)), err_sum / max(err_count, 1)


vit.train()
head.train()

for epoch in range(STARTING_EPOCH, NUM_EPOCHS):
    losses, sigreg_losses = [], []
    train_err_sum, train_err_count = torch.zeros(TARGET_DIM), 0

    for frame_i, frame_j, target in tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}', leave=False):
        frame_i, frame_j = to_float(frame_i), to_float(frame_j)
        target      = target.to(device)
        target_norm = normalize_target(target)

        emb  = extract_embedding(frame_i, frame_j)
        pred = head(emb)                                       # (B, 2) normalised
        h    = huber_loss_fn(pred, target_norm)
        sreg = sigreg_loss(emb, gamma=SIGREG_GAMMA)
        loss = h + SIGREG_WEIGHT * sreg

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(vit.parameters()) + list(head.parameters()), max_norm=5.0)
        optimizer.step()

        pred_raw = denormalize_target(pred.detach())
        e_sum, e_cnt = abs_error_sum(pred_raw, target)
        train_err_sum += e_sum; train_err_count += e_cnt
        losses.append(h.item())
        sigreg_losses.append(sreg.item())

    scheduler.step()
    val_loss, val_abs_err = run_validation()
    train_abs_err = train_err_sum / max(train_err_count, 1)
    train_err_str = '  '.join(f'{n}={train_abs_err[i]:.4f}{u}' for i, (n, u) in enumerate(zip(AXIS_NAMES, AXIS_UNITS)))
    val_err_str   = '  '.join(f'{n}={val_abs_err[i]:.4f}{u}' for i, (n, u) in enumerate(zip(AXIS_NAMES, AXIS_UNITS)))
    print(f'Epoch {epoch+1}/{NUM_EPOCHS}  '
          f'train_loss={np.mean(losses):.4f}  sigreg={np.mean(sigreg_losses):.4f}  '
          f'lr={scheduler.get_last_lr()[-1]:.2e}')
    print(f'  train|err|: {train_err_str}')
    print(f'  val_loss={val_loss:.4f}  val|err|: {val_err_str}')

    if (epoch + 1) % 10 == 0 or epoch + 1 == NUM_EPOCHS:
        torch.save({
            'epoch': epoch + 1,
            'vit': vit.state_dict(),
            'head': head.state_dict(),
            'delta_mean': delta_mean.cpu(),
            'delta_std': delta_std.cpu(),
        }, CKPT_DIR / f'diff_pred_robust_epoch_{epoch+1:04d}.pt')

## Testing
### Per-axis error distribution — 100 random synthetic samples
Same format as `diff_pred_20x.ipynb`'s test cell, simplified to 2 axes (no masking).

In [ ]:
N_TEST = 100
_rng_test = np.random.default_rng(7)

test_preds, test_gt = [], []
vit.eval()
head.eval()
with torch.no_grad():
    for _ in tqdm(range(N_TEST), desc='Testing'):
        i1, i2 = _rng_test.choice(len(val_paths), size=2, replace=False)
        frame_i, frame_j, target_mm, _debug = make_pair_sample(val_paths[i1], val_paths[i2], _rng_test)

        fi_t = to_float(torch.from_numpy(frame_i[:, :, ::-1].copy()).permute(2, 0, 1).unsqueeze(0))
        fj_t = to_float(torch.from_numpy(frame_j[:, :, ::-1].copy()).permute(2, 0, 1).unsqueeze(0))
        emb = extract_embedding(fi_t, fj_t)
        pred_raw = denormalize_target(head(emb)).squeeze(0).cpu().numpy()

        test_preds.append(pred_raw)
        test_gt.append(target_mm)
vit.train()
head.train()

test_preds = np.stack(test_preds)
test_gt    = np.stack(test_gt)
errors     = test_preds - test_gt

print(f'Per-axis MAE — {N_TEST} random synthetic samples from the val pool:')
for i, (name, unit) in enumerate(zip(AXIS_NAMES, AXIS_UNITS)):
    mae  = np.abs(errors[:, i]).mean()
    rmse = np.sqrt((errors[:, i] ** 2).mean())
    print(f'  {name}: MAE={mae:.3f} {unit}   RMSE={rmse:.3f} {unit}')

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for i, (name, unit) in enumerate(zip(AXIS_NAMES, AXIS_UNITS)):
    ax = axes[i]
    gt = test_gt[:, i]
    pr = test_preds[:, i]
    lim = max(np.abs(gt).max(), np.abs(pr).max()) * 1.05
    ax.scatter(gt, pr, s=8, alpha=0.6, color='steelblue')
    ax.plot([-lim, lim], [-lim, lim], 'r--', lw=1.2, label='ideal')
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_xlabel(f'GT Δ{name} ({unit})'); ax.set_ylabel(f'Pred Δ{name} ({unit})')
    ax.set_title(f'Δ{name}  MAE={np.abs(errors[:, i]).mean():.3f} {unit}')
    ax.set_aspect('equal')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### Per-axis error distribution — `train_loader` and `val_loader`

In [ ]:
train_preds, train_gt = [], []

vit.eval()
head.eval()
with torch.no_grad():
    for frame_i, frame_j, target in tqdm(train_loader, desc='Evaluating (train)'):
        frame_i = to_float(frame_i.to(device))
        frame_j = to_float(frame_j.to(device))
        emb = extract_embedding(frame_i, frame_j)
        pred_norm = head(emb)
        pred_raw  = denormalize_target(pred_norm).cpu()
        train_preds.append(pred_raw)
        train_gt.append(target)
vit.train()
head.train()

train_preds = torch.cat(train_preds).numpy()
train_gt    = torch.cat(train_gt).numpy()
errors      = train_preds - train_gt

print(f'Per-axis MAE — {len(train_preds):,} train_loader pairs:')
for i, (name, unit) in enumerate(zip(AXIS_NAMES, AXIS_UNITS)):
    mae  = np.abs(errors[:, i]).mean()
    rmse = np.sqrt((errors[:, i] ** 2).mean())
    print(f'  {name}: MAE={mae:.3f} {unit}   RMSE={rmse:.3f} {unit}')

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for i, (name, unit) in enumerate(zip(AXIS_NAMES, AXIS_UNITS)):
    ax = axes[i]
    gt = train_gt[:, i]
    pr = train_preds[:, i]
    lim = max(np.abs(gt).max(), np.abs(pr).max()) * 1.05
    ax.scatter(gt, pr, s=2, alpha=0.3, color='steelblue', rasterized=True)
    ax.plot([-lim, lim], [-lim, lim], 'r--', lw=1.2, label='ideal')
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_xlabel(f'GT Δ{name} ({unit})'); ax.set_ylabel(f'Pred Δ{name} ({unit})')
    ax.set_title(f'Δ{name}  MAE={np.abs(errors[:, i]).mean():.3f} {unit}')
    ax.set_aspect('equal')
    ax.legend(fontsize=8)
plt.suptitle('train_loader')
plt.tight_layout()
plt.show()

In [ ]:
val_preds, val_gt = [], []

vit.eval()
head.eval()
with torch.no_grad():
    for frame_i, frame_j, target in tqdm(val_loader, desc='Evaluating (val)'):
        frame_i = to_float(frame_i.to(device))
        frame_j = to_float(frame_j.to(device))
        emb = extract_embedding(frame_i, frame_j)
        pred_norm = head(emb)
        pred_raw  = denormalize_target(pred_norm).cpu()
        val_preds.append(pred_raw)
        val_gt.append(target)
vit.train()
head.train()

val_preds = torch.cat(val_preds).numpy()
val_gt    = torch.cat(val_gt).numpy()
errors    = val_preds - val_gt

print(f'Per-axis MAE — {len(val_preds):,} val_loader pairs:')
for i, (name, unit) in enumerate(zip(AXIS_NAMES, AXIS_UNITS)):
    mae  = np.abs(errors[:, i]).mean()
    rmse = np.sqrt((errors[:, i] ** 2).mean())
    print(f'  {name}: MAE={mae:.3f} {unit}   RMSE={rmse:.3f} {unit}')

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for i, (name, unit) in enumerate(zip(AXIS_NAMES, AXIS_UNITS)):
    ax = axes[i]
    gt = val_gt[:, i]
    pr = val_preds[:, i]
    lim = max(np.abs(gt).max(), np.abs(pr).max()) * 1.05
    ax.scatter(gt, pr, s=2, alpha=0.3, color='steelblue', rasterized=True)
    ax.plot([-lim, lim], [-lim, lim], 'r--', lw=1.2, label='ideal')
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_xlabel(f'GT Δ{name} ({unit})'); ax.set_ylabel(f'Pred Δ{name} ({unit})')
    ax.set_title(f'Δ{name}  MAE={np.abs(errors[:, i]).mean():.3f} {unit}')
    ax.set_aspect('equal')
    ax.legend(fontsize=8)
plt.suptitle('val_loader')
plt.tight_layout()
plt.show()

### Qualitative check — one random val-pool sample

In [ ]:
_rng_vis = np.random.default_rng()
_i1, _i2 = _rng_vis.choice(len(val_paths), size=2, replace=False)
vis_bottom_path, vis_top_path = val_paths[_i1], val_paths[_i2]
vis_frame_i, vis_frame_j, vis_target_mm, vis_debug = make_pair_sample(vis_bottom_path, vis_top_path, _rng_vis)

vit.eval()
head.eval()
with torch.no_grad():
    fi_t = torch.from_numpy(vis_frame_i[:, :, ::-1].copy()).permute(2, 0, 1).unsqueeze(0)
    fj_t = torch.from_numpy(vis_frame_j[:, :, ::-1].copy()).permute(2, 0, 1).unsqueeze(0)
    emb  = extract_embedding(to_float(fi_t), to_float(fj_t))
    vis_pred_raw = denormalize_target(head(emb)).squeeze(0).cpu().numpy()
vit.train()
head.train()

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(cv2.cvtColor(vis_frame_i, cv2.COLOR_BGR2RGB)); axes[0].set_title('frame_i (composite 1)'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(vis_frame_j, cv2.COLOR_BGR2RGB)); axes[1].set_title(f"frame_j (composite 2, opacity {vis_debug['opacity']:.3f})"); axes[1].axis('off')

gt_str   = '  '.join(f'{n}={vis_target_mm[i]:+.4f}{u}' for i, (n, u) in enumerate(zip(AXIS_NAMES, AXIS_UNITS)))
pred_str = '  '.join(f'{n}={vis_pred_raw[i]:+.4f}{u}' for i, (n, u) in enumerate(zip(AXIS_NAMES, AXIS_UNITS)))
plt.suptitle(f'bottom={vis_bottom_path.name}  top={vis_top_path.name}\nGT:   {gt_str}\nPred: {pred_str}', fontsize=10)
plt.tight_layout()
plt.show()

### Qualitative check — synthetic 10x composite vs. real 20x frame (cross-domain)

Runs the trained model on `_synthetic_5`/`_real_20x_resized_5` from the SIFT cross-check sanity
cell above (`folder 5`), feeding the real video frame as `frame_i` and the synthetic composite as
`frame_j`. There's no defined ground-truth target for this pairing — the real frame isn't a second
window position of the same composite, just an independent real capture — so this is a
qualitative-only look at how the model (trained purely on synthetic 10x data) responds to a real
20x input, not an error metric.

In [ ]:
vit.eval()
head.eval()
with torch.no_grad():
    fi_t5 = torch.from_numpy(_real_20x_resized_5[:, :, ::-1].copy()).permute(2, 0, 1).unsqueeze(0)
    fj_t5 = torch.from_numpy(_synthetic_5[:, :, ::-1].copy()).permute(2, 0, 1).unsqueeze(0)
    emb5 = extract_embedding(to_float(fi_t5), to_float(fj_t5))
    pred_raw_5 = denormalize_target(head(emb5)).squeeze(0).cpu().numpy()
vit.train()
head.train()

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(cv2.cvtColor(_real_20x_resized_5, cv2.COLOR_BGR2RGB)); axes[0].set_title('frame_i (real 20x.mp4 frame)'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(_synthetic_5, cv2.COLOR_BGR2RGB)); axes[1].set_title('frame_j (synthetic 10x composite)'); axes[1].axis('off')

pred_str_5 = '  '.join(f'{n}={pred_raw_5[i]:+.4f}{u}' for i, (n, u) in enumerate(zip(AXIS_NAMES, AXIS_UNITS)))
plt.suptitle(f'Cross-domain prediction (no ground truth for this pairing)\nPred: {pred_str_5}', fontsize=10)
plt.tight_layout()
plt.show()
